# Комплексный анализ динамики модели Daisyworld

В данном скрипте исследуется поведение модели Daisyworld при изменении
солнечной светимости (сценарий :ramp). Строится комплексный график,
включающий:
- динамику численности чёрных и белых маргариток
- изменение средней температуры поверхности
- изменение солнечной светимости

## Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"

using Agents
using DataFrames
using Plots

### Подключение модели

Импортируем определение модели Daisyworld из исходного файла.

In [ ]:
include(srcdir("daisyworld.jl"))

### Настройка визуализации

Для построения графиков используется библиотека CairoMakie,
обеспечивающая высокое качество отображения.

In [ ]:
using CairoMakie

## Определение агрегатных функций

Для сбора статистики о популяции маргариток определим две функции,
которые проверяют принадлежность агента к определённому виду:
- `black(a)` — возвращает `true`, если маргаритка чёрная
- `white(a)` — возвращает `true`, если маргаритка белая

In [ ]:
black(a) = a.breed == :black
white(a) = a.breed == :white

### Агрегатные данные об агентах

`adata` определяет, какие данные о агентах будут собираться в процессе
моделирования. Здесь мы собираем количество чёрных и белых маргариток
на каждом шаге.

In [ ]:
adata = [(black, count), (white, count)]

## Создание модели со сценарием изменения светимости

Создаём модель с начальной солнечной светимостью 1.0 и сценарием :ramp.
При сценарии :ramp солнечная светимость изменяется по следующему закону:
- увеличивается на 0.005 с шага 200 по 400
- уменьшается на 0.0025 с шага 500 по 750

In [ ]:
model = daisyworld(; solar_luminosity = 1.0, scenario = :ramp)

### Агрегатные данные о модели

`temperature(model)` — функция, вычисляющая среднюю температуру
по всем клеткам модели.

In [ ]:
temperature(model) = StatsBase.mean(model.temperature)

### Данные для сбора о модели

`mdata` определяет, какие данные о модели будут собираться:
- средняя температура поверхности
- текущая солнечная светимость

In [ ]:
mdata = [temperature, :solar_luminosity]

## Запуск модели

Запускаем симуляцию на 1000 шагов. Результаты сохраняются в два DataFrame:
- `agent_df` — данные об агентах (количество чёрных и белых маргариток)
- `model_df` — данные о модели (температура и светимость)

In [ ]:
agent_df, model_df = run!(model, 1000; adata = adata, mdata = mdata)

## Построение комплексного графика

### Создание фигуры

Создаём фигуру размером 600×600 пикселей с тремя вертикальными осями.

In [ ]:
figure = CairoMakie.Figure(size = (600, 600))

### Верхний график: численность маргариток

На первом графике отображается динамика численности маргариток:
- чёрные маргаритки — красная линия
- белые маргаритки — синяя линия

In [ ]:
ax1 = figure[1, 1] = Axis(figure, ylabel = "daisy count")

blackl = lines!(ax1,
    agent_df[!, :time],
    agent_df[!, :count_black],
    color = :red
)

whitel = lines!(ax1,
    agent_df[!, :time],
    agent_df[!, :count_white],
    color = :blue
)

### Легенда

Добавляем легенду для идентификации линий на верхнем графике.

In [ ]:
figure[1, 2] = Legend(figure, [blackl, whitel], ["black", "white"])

### Средний график: температура

На втором графике отображается изменение средней температуры
поверхности во времени.

In [ ]:
ax2 = figure[2, 1] = Axis(figure, ylabel = "temperature")

lines!(ax2,
    model_df[!, :time],
    model_df[!, :temperature],
    color = :red
)

### Нижний график: солнечная светимость

На третьем графике отображается изменение солнечной светимости
в соответствии со сценарием :ramp.

In [ ]:
ax3 = figure[3, 1] = Axis(figure,
    xlabel = "tick",
    ylabel = "luminosity"
)

lines!(ax3,
    model_df[!, :time],
    model_df[!, :solar_luminosity],
    color = :red
)

### Скрытие меток времени на верхних графиках

Для улучшения читаемости графика скрываем метки времени
на верхних двух графиках (они отображаются только на нижнем).

In [ ]:
for ax in (ax1, ax2)
    ax.xticklabelsvisible = false
end

## Отображение и сохранение результата

Отображаем фигуру (при необходимости) и сохраняем её в каталог `plots/`.

In [ ]:
figure
save(plotsdir("daisy_luminosity.png"), figure)

## Интерпретация результатов

На полученном комплексном графике можно наблюдать,
как система Daisyworld реагирует на изменение внешнего воздействия:

1. **Фаза увеличения светимости (tick 200–400)**:
   - Солнечная светимость растёт
   - Температура повышается
   - Белые маргаритки получают преимущество (отражают свет, охлаждая среду)
   - Чёрные маргаритки сокращаются

2. **Фаза стабилизации (tick 400–500)**:
   - Система адаптируется к новым условиям
   - Устанавливается новое равновесие

3. **Фаза уменьшения светимости (tick 500–750)**:
   - Солнечная светимость снижается
   - Температура падает
   - Чёрные маргаритки получают преимущество (поглощают свет, нагревая среду)
   - Белые маргаритки сокращаются

### Ключевой вывод

Модель демонстрирует способность биосферы к саморегуляции:
несмотря на значительные изменения внешнего воздействия (солнечной светимости),
система поддерживает температуру в диапазоне, благоприятном для жизни,
за счёт изменения соотношения чёрных и белых маргариток.